In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy
from tabulate import tabulate
from typing import Dict, Tuple, Optional
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# ==========================================
# 1. CONFIGURATION & UTILS
# ==========================================
class Config:
    IMG_SIZE = (512, 512)
    # Segmentation sensitivity
    SEG_THRESH_ADAPTIVE_BLOCK = 11
    SEG_THRESH_ADAPTIVE_C = 2
    # Feature Normalization (for 0-1 scores)
    NORM_ASYMMETRY = 0.3    # Expected max asymmetry
    NORM_BORDER = 4.0       # Expected max irregularity
    NORM_COLOR = 60.0       # Expected max color deviation
    NORM_TEXTURE = 5.0      # Expected max entropy

def display_error(msg):
    print(f"❌ ERROR: {msg}")

# ==========================================
# 2. IMAGE PROCESSING PIPELINE
# ==========================================
class ImageProcessor:
    @staticmethod
    def load_and_prep(path: str) -> Tuple[np.ndarray, np.ndarray]:
        """Loads image, checks quality, resizes, and applies CLAHE."""
        img = cv2.imread(path)
        if img is None:
            raise ValueError("Image not found or corrupted.")
        
        # Resize
        img = cv2.resize(img, Config.IMG_SIZE, interpolation=cv2.INTER_LANCZOS4)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Contrast Limited Adaptive Histogram Equalization (CLAHE)
        # Helps normalize lighting conditions
        lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        cl = clahe.apply(l)
        limg = cv2.merge((cl,a,b))
        img_enhanced = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
        
        return img_rgb, img_enhanced

    @staticmethod
    def remove_hair(img: np.ndarray) -> np.ndarray:
        """DullRazor algorithm: BlackHat transform + Telea Inpainting."""
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17))
        blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
        
        # Create hair mask
        _, mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
        mask = cv2.dilate(mask, np.ones((3,3), np.uint8), iterations=1)
        
        # Inpaint
        img_hair_free = cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA)
        return img_hair_free

    @staticmethod
    def segment_lesion(img: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Robust segmentation using combined Adaptive Thresholding + HSV."""
        # 1. Adaptive Thresholding (Structure)
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        adaptive = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                       cv2.THRESH_BINARY_INV, 21, 4)
        
        # 2. HSV Thresholding (Color) - Detects skin anomalies
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
        # Lower mask (Redish)
        lower1 = np.array([0, 50, 50])
        upper1 = np.array([20, 255, 255])
        # Upper mask (Redish/Purple)
        lower2 = np.array([160, 50, 50])
        upper2 = np.array([180, 255, 255])
        
        mask1 = cv2.inRange(hsv, lower1, upper1)
        mask2 = cv2.inRange(hsv, lower2, upper2)
        hsv_mask = cv2.add(mask1, mask2)
        
        # 3. Combine: Use 'OR' logic to catch everything, then clean up
        combined = cv2.bitwise_or(adaptive, hsv_mask)
        
        # 4. Morphological Cleanup (Remove noise, fill holes)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        combined = cv2.morphologyEx(combined, cv2.MORPH_OPEN, kernel, iterations=2)
        combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE, kernel, iterations=2)
        
        # 5. Keep only the largest contour
        contours, _ = cv2.findContours(combined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        final_mask = np.zeros_like(combined)
        
        largest_cnt = None
        if contours:
            largest_cnt = max(contours, key=cv2.contourArea)
            # Filter: If contour is too small, ignore it
            if cv2.contourArea(largest_cnt) > 500:
                cv2.drawContours(final_mask, [largest_cnt], -1, 255, -1)
            else:
                largest_cnt = None
                
        return final_mask, largest_cnt

# ==========================================
# 3. FEATURE EXTRACTION ENGINE (UPDATED)
# ==========================================
class FeatureExtractor:
    
    @staticmethod
    def analyze_asymmetry(mask: np.ndarray) -> Tuple[float, np.ndarray]:
        """
        Rotation-Invariant Asymmetry.
        Aligns lesion vertically using moments, then flips to check overlap.
        """
        if np.sum(mask) == 0: return 0.0, mask
        
        # 1. Calculate Moments & Orientation
        M = cv2.moments(mask)
        if M['m00'] == 0: return 0.0, mask
        
        cx, cy = int(M['m10']/M['m00']), int(M['m01']/M['m00'])
        mu11, mu20, mu02 = M['mu11'], M['mu20'], M['mu02']
        theta = 0.5 * np.arctan2(2 * mu11, mu20 - mu02) # Radians
        angle = np.degrees(theta)
        
        # 2. Rotate Mask to align major axis vertically
        h, w = mask.shape
        rot_mat = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
        rotated_mask = cv2.warpAffine(mask, rot_mat, (w, h), flags=cv2.INTER_NEAREST)
        
        # 3. Recenter mask (Crucial for flipping)
        M_rot = cv2.moments(rotated_mask)
        if M_rot['m00'] != 0:
            new_cx = int(M_rot['m10']/M_rot['m00'])
            shift_x = w//2 - new_cx
            M_trans = np.float32([[1, 0, shift_x], [0, 1, 0]])
            centered_mask = cv2.warpAffine(rotated_mask, M_trans, (w, h))
        else:
            centered_mask = rotated_mask

        # 4. Calculate Asymmetry (IoU Logic)
        # Flip Left-Right
        flip_lr = cv2.flip(centered_mask, 1)
        xor_img = cv2.bitwise_xor(centered_mask, flip_lr)
        
        area_lesion = np.sum(centered_mask > 0)
        area_diff = np.sum(xor_img > 0)
        
        # Score: How much of the area does NOT overlap?
        score = area_diff / area_lesion if area_lesion > 0 else 0
        
        # Return score and the alignment visualization
        viz_img = np.zeros((h, w, 3), dtype=np.uint8)
        viz_img[centered_mask > 0] = [100, 100, 100] # Grey lesion
        viz_img[xor_img > 0] = [255, 50, 50]         # Red Difference
        
        return score, viz_img

    @staticmethod
    def analyze_border(contour: np.ndarray) -> Tuple[float, float, float]:
        """
        Analyzes Jaggedness using Solidity (Convex Hull).
        Returns: (Score, Compactness, Solidity)
        """
        if contour is None: return 0.0, 0.0, 0.0
        
        area = cv2.contourArea(contour)
        perimeter = cv2.arcLength(contour, True)
        if area == 0: return 0.0, 0.0, 0.0
        
        # 1. Compactness (General Shape)
        compactness = (perimeter ** 2) / (4 * np.pi * area)
        
        # 2. Solidity (Jaggedness / "Bites")
        hull = cv2.convexHull(contour)
        hull_area = cv2.contourArea(hull)
        solidity = area / hull_area if hull_area > 0 else 1.0
        
        # 3. Combined Metric
        # High Compactness = Irregular Shape
        # Low Solidity = Jagged Borders
        # We invert solidity so higher = worse
        score = (compactness * 0.3) + ((1 - solidity) * 5.0)
        
        return score, compactness, solidity

    @staticmethod
    def analyze_color(img: np.ndarray, mask: np.ndarray) -> float:
        """
        Analyzes Color Variegation using CIELab Standard Deviation.
        """
        if np.sum(mask) == 0: return 0.0
        
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        
        # Mask out background
        mask_bool = mask > 0
        l = lab[:,:,0][mask_bool]
        a = lab[:,:,1][mask_bool]
        b = lab[:,:,2][mask_bool]
        
        # Calculate Standard Deviation (Spread of colors)
        std_l = np.std(l)
        std_a = np.std(a)
        std_b = np.std(b)
        
        # Combined score
        color_std_sum = std_l + std_a + std_b
        return color_std_sum

    @staticmethod
    def analyze_texture(img: np.ndarray, mask: np.ndarray) -> float:
        """
        Analyzes internal texture chaos using Entropy on MASKED pixels.
        """
        if np.sum(mask) == 0: return 0.0
        
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        pixels = gray[mask > 0]
        
        # Histogram of intensities
        hist, _ = np.histogram(pixels, bins=256, range=(0,256))
        prob_dist = hist / hist.sum()
        
        # Entropy (Randomness)
        # High entropy = chaotic texture (Melanoma)
        # Low entropy = smooth texture (Benign)
        ent = entropy(prob_dist, base=2)
        return ent

# ==========================================
# 4. MASTER CONTROLLER
# ==========================================
def run_analysis(image_path):
    try:
        # A. Load & Preprocess
        original, enhanced = ImageProcessor.load_and_prep(image_path)
        hair_free = ImageProcessor.remove_hair(enhanced)
        
        # B. Segmentation
        mask, contour = ImageProcessor.segment_lesion(hair_free)
        
        if contour is None:
            raise ValueError("No lesion detected. Image might be too blurry or low contrast.")
            
        # C. Feature Extraction (Calculation happens HERE only)
        asym_score, asym_viz = FeatureExtractor.analyze_asymmetry(mask)
        bord_score, compact, solid = FeatureExtractor.analyze_border(contour)
        color_score = FeatureExtractor.analyze_color(hair_free, mask)
        text_score = FeatureExtractor.analyze_texture(hair_free, mask)
        
        # D. Create Overlay for Viz
        overlay = hair_free.copy()
        cv2.drawContours(overlay, [contour], -1, (0, 255, 0), 2)
        
        # E. Store Results in Single Dictionary
        results = {
            "images": {
                "orig": original,
                "clean": hair_free,
                "mask": mask,
                "overlay": overlay,
                "asym_viz": asym_viz
            },
            "metrics": {
                "asymmetry": asym_score,
                "border_score": bord_score,
                "compactness": compact,
                "solidity": solid,
                "color_std": color_score,
                "texture_entropy": text_score
            },
            "normalized": {
                "A": min(1.0, asym_score / Config.NORM_ASYMMETRY),
                "B": min(1.0, bord_score / Config.NORM_BORDER),
                "C": min(1.0, color_score / Config.NORM_COLOR),
                "T": min(1.0, text_score / Config.NORM_TEXTURE)
            }
        }
        return results
        
    except Exception as e:
        display_error(str(e))
        return None

# ==========================================
# 5. VISUALIZATION & REPORTING
# ==========================================
def display_dashboard(results):
    if results is None: return

    imgs = results["images"]
    mets = results["metrics"]
    norm = results["normalized"]
    
    # --- 1. PLOTTING ---
    fig = plt.figure(figsize=(15, 10))
    gs = fig.add_gridspec(2, 4)
    
    # Top Row: Images
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.imshow(imgs["orig"])
    ax1.set_title("Original Image")
    ax1.axis("off")
    
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.imshow(imgs["clean"])
    ax2.set_title("Hair Removed & Enhanced")
    ax2.axis("off")
    
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.imshow(imgs["mask"], cmap='gray')
    ax3.set_title("Segmentation Mask")
    ax3.axis("off")
    
    ax4 = fig.add_subplot(gs[0, 3])
    ax4.imshow(imgs["overlay"])
    ax4.set_title("Detected Contour (Green)")
    ax4.axis("off")
    
    # Bottom Row: Feature Visuals
    
    # Asymmetry Viz
    ax5 = fig.add_subplot(gs[1, 0])
    ax5.imshow(imgs["asym_viz"])
    ax5.set_title(f"Asymmetry Alignment\n(Red = Diff) Score: {mets['asymmetry']:.2f}")
    ax5.axis("off")
    
    # Color Analysis Bar
    ax6 = fig.add_subplot(gs[1, 1])
    ax6.bar(["L (Light)", "a (Grn-Red)", "b (Blu-Yel)"], [0,0,0], color=['gray', 'red', 'gold']) # Placeholder visual
    ax6.text(0.5, 0.5, f"Color Variegation\nStdDev Sum: {mets['color_std']:.1f}", 
             ha='center', va='center', fontsize=12)
    ax6.set_title("Color Analysis")
    ax6.axis("off")

    # ABCT Score Bars (The "Final Verdict")
    ax7 = fig.add_subplot(gs[1, 2:])
    labels = ['Asymmetry', 'Border (Jagged)', 'Color (Mix)', 'Texture (Chaos)']
    values = [norm['A'], norm['B'], norm['C'], norm['T']]
    colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99']
    
    bars = ax7.bar(labels, values, color=colors)
    ax7.set_ylim(0, 1.1)
    ax7.axhline(y=0.5, color='r', linestyle='--', alpha=0.3, label="Risk Threshold")
    ax7.set_title("Normalized ABCD Feature Scores (0.0 = Safe, 1.0 = High Risk)")
    
    # Add exact values on top of bars
    for bar in bars:
        height = bar.get_height()
        ax7.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.show()
    
    # --- 2. TABULAR REPORT ---
    # Ensures values match the plots exactly
    table_data = [
        ["Feature", "Raw Value", "Normalized Risk (0-1)", "Interpretation"],
        ["Asymmetry", f"{mets['asymmetry']:.4f}", f"{norm['A']:.2f}", "Shape Skew"],
        ["Border", f"{mets['border_score']:.4f}", f"{norm['B']:.2f}", "Jaggedness"],
        ["Color", f"{mets['color_std']:.4f}", f"{norm['C']:.2f}", "Color Mixing"],
        ["Texture", f"{mets['texture_entropy']:.4f}", f"{norm['T']:.2f}", "Internal Chaos"]
    ]
    
    print("\n" + "="*60)
    print("🔬 CLINICAL FEATURE EXTRACTION REPORT")
    print("="*60)
    print(tabulate(table_data, headers="firstrow", tablefmt="fancy_grid"))
    
    # Simple Total Risk Calculation (Average of A, B, C, T)
    total_risk = (norm['A'] + norm['B'] + norm['C'] + norm['T']) / 4.0
    print(f"\n📊 ESTIMATED AGGREGATE RISK SCORE: {total_risk:.2f} / 1.0")
    if total_risk > 0.5:
        print("⚠️  RESULT: High probability of features associated with Melanoma.")
    else:
        print("✅ RESULT: Features appear consistent with Benign lesions.")
    print("="*60)

# ==========================================
# 6. EXECUTION
# ==========================================
# Replace this path with your specific image file
img_path = r"C:\Users\Aakarsh Goyal\Downloads\archive\PH2Dataset\PH2 Dataset images\IMD427\IMD427_Dermoscopic_Image\IMD427.bmp" 

# 1. Run Analysis
analysis_result = run_analysis(img_path)

# 2. Display Dashboard
display_dashboard(analysis_result)

1. Loading Image...
2. Removing Hair...


AttributeError: 'tuple' object has no attribute 'size'